**Contact Boundary and Gap Function in NGSolve**

In [84]:

from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

**Parallel Contact Boundary**

In [85]:
f1 = MoveTo(0, 0).Rectangle(1, 1).Face()
f1.edges.Max(Y).name = "contact_1"
f1.name = "body_1"

f2 = MoveTo(0.0, 1).Rectangle(1, 1).Face()
f2 = f2.Rotate(Axis((0.5, 1.5, 0), (0, 0, 1)), 90)
f2.edges.Min(Y).name = "contact_2"
f2.name = "body_2"

geo = Compound([f1, f2])
geo = OCCGeometry(geo, dim=2)

mesh = Mesh(geo.GenerateMesh(maxh=0.1, quad_dominated=True))

Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
fes = VectorH1(mesh, order=2, dirichlet="left|right|bottom|top")
u = fes.TrialFunction()
v = fes.TestFunction()

# Define a displacement field
offset = -0.1
gfu = GridFunction(fes)
gfu.Set((0, offset), definedon=mesh.Materials("body_2"))
Draw(gfu, mesh, "displacement", deformation=gfu)

# Define Contact
master = mesh.Boundaries("contact_1")
slave = mesh.Boundaries("contact_2")
contact = ContactBoundary(master, slave)

# Update contact with current displacement
contact.Update(gfu, None, 10, 1)

# Get gap vector and normal
gap_vec_master = contact.gap
n_master = contact.normal

gap_n_master = InnerProduct(gap_vec_master, n_master)

# Calculate the gap distance
gap_integral = Integrate(gap_n_master, master)
print(f"Gap distance: {gap_integral:.6f}")

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Gap distance: -0.100000


For parallel contact boundaries this approach yields the correct result.

-----------------------------------------------------------

**Curved Contact Boundary**

In [88]:
square = MoveTo(-0.5, 0).Rectangle(1, 1).Face()
square.edges.Max(Y).name = "contact_square"
square.name = "square"

circle = Circle((0, 1.5), 0.5).Face()
circle.edges.name = "contact_circle"
circle.name = "circle"

geo = Compound([square, circle])
geo = OCCGeometry(geo, dim=2)

mesh = Mesh(geo.GenerateMesh(maxh=0.1, quad_dominated=False))

Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [89]:
fes = VectorH1(mesh, order=3)
u = fes.TrialFunction()
v = fes.TestFunction()

offset = -0.1

gfu = GridFunction(fes)
gfu.Set((0, offset), definedon=mesh.Materials("circle"))

Draw(gfu, mesh, "displacement", deformation=gfu)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [90]:
# Define Contact
master = mesh.Boundaries("contact_square")
slave = mesh.Boundaries("contact_circle")
contact = ContactBoundary(master, slave)

# Update contact with current displacement
contact.Update(gfu, None, 10, 1)

In [92]:
# Get gap vector and normal
gap_vec_master = contact.gap
n_master = contact.normal

gap_n_master = InnerProduct(gap_vec_master, n_master)

# Calculate the gap distance
gap_integral = Integrate(gap_n_master, master)
print(f"Gap distance: {gap_integral}")

Gap distance: -0.017480268959969618


Inner product of gao_vec_mastera and gap normal vector is integrated over the total master contact boundary.
Since the circle penetrate the square only on a small part of the master boundary, the gap integral does not correspond to the actual maximum penetration depth.

In [ ]:
# Method 1: Using gap coefficient function directly
def calculate_penetration_with_coeff_func(contact, mesh_obj, boundary_name):
    """Calculate penetration using coefficient functions"""

    # Get the gap coefficient function from contact
    gap_vec = contact.gap
    n_master = contact.normal
    gap_n = InnerProduct(gap_vec, n_master)

    # Create a finite element space for the gap function
    gap_fes = H1(mesh_obj, order=2)
    gap_gf = GridFunction(gap_fes)

    # Project the gap function onto the finite element space
    gap_gf.Set(gap_n, definedon=mesh_obj.Boundaries(boundary_name))

    # Calculate statistics
    # For penetration, we're interested in negative values
    penetration_cf = IfPos(-gap_n, -gap_n, 0)  # Only negative gaps (penetration)
    gap_positive_cf = IfPos(gap_n, gap_n, 0)  # Only positive gaps (separation)

    # Integrate over boundary
    max_penetration_integral = Integrate(penetration_cf, mesh_obj.Boundaries(boundary_name))
    positive_gap_integral = Integrate(gap_positive_cf, mesh_obj.Boundaries(boundary_name))
    total_gap_integral = Integrate(gap_n, mesh_obj.Boundaries(boundary_name))
    boundary_length = Integrate(CF(1), mesh_obj.Boundaries(boundary_name))

    # Average values
    avg_penetration = max_penetration_integral / boundary_length if boundary_length > 0 else 0
    avg_positive_gap = positive_gap_integral / boundary_length if boundary_length > 0 else 0
    avg_total_gap = total_gap_integral / boundary_length if boundary_length > 0 else 0

    return {
        "gap_coefficient_function": gap_n,
        "penetration_cf": penetration_cf,
        "avg_penetration": avg_penetration,
        "avg_positive_gap": avg_positive_gap,
        "avg_total_gap": avg_total_gap,
        "total_penetration_integral": max_penetration_integral,
        "boundary_length": boundary_length,
    }


# Usage example:
penetration_info = calculate_penetration_with_coeff_func(contact, mesh, "contact_square")
print(f"Average penetration: {penetration_info['avg_penetration']:.6f}")
print(f"Average positive gap: {penetration_info['avg_positive_gap']:.6f}")
print(f"Average total gap: {penetration_info['avg_total_gap']:.6f}")

In [ ]:
# Method 2: Find maximum penetration using coefficient function evaluation
def find_max_penetration_coeff(gap_cf, mesh_obj, boundary_name, num_samples=100):
    """Find maximum penetration by sampling the coefficient function"""

    max_penetration = 0
    min_gap = float("inf")

    # Sample points along the boundary
    for el in mesh_obj.Elements(BND):
        if el.mat == boundary_name:
            # Get element transformation
            trafo = mesh_obj.GetTrafo(el)

            # Create sampling points along element
            for i in range(num_samples):
                # For 1D boundary elements, sample along reference element
                if el.type == ET.SEGM:  # 1D boundary element
                    xi = -1 + 2 * i / (num_samples - 1)  # Sample from -1 to 1
                    ip = IntegrationPoint((xi, 0, 0))
                else:
                    continue

                try:
                    # Map to physical coordinates
                    mapped_ip = trafo(ip)

                    # Evaluate gap function
                    gap_val = gap_cf(mapped_ip)
                    min_gap = min(min_gap, gap_val)

                    # Maximum penetration is absolute value of most negative gap
                    if gap_val < 0:
                        max_penetration = max(max_penetration, abs(gap_val))

                except:
                    continue

    return max_penetration, min_gap


# Usage:
gap_cf = gap_n_master  # From your existing code
max_pen, min_gap = find_max_penetration_coeff(gap_cf, mesh, "contact_square")
print(f"Maximum penetration (sampling): {max_pen:.6f}")
print(f"Minimum gap value: {min_gap:.6f}")

In [ ]:
# Method 3: Comprehensive coefficient function approach with visualization
def analyze_contact_gap_comprehensive(contact, mesh_obj, boundary_name):
    """Comprehensive gap analysis using coefficient functions"""

    # Get gap components
    gap_vec = contact.gap
    n_master = contact.normal
    gap_n = InnerProduct(gap_vec, n_master)

    # Create different coefficient functions for analysis
    penetration_cf = IfPos(-gap_n, -gap_n, 0)  # Penetration depth (positive when penetrating)
    separation_cf = IfPos(gap_n, gap_n, 0)  # Separation distance (positive when separated)
    gap_magnitude_cf = sqrt(gap_n * gap_n)  # Absolute gap magnitude

    # Integrate over boundary
    boundary_region = mesh_obj.Boundaries(boundary_name)

    # Calculate various metrics
    total_penetration = Integrate(penetration_cf, boundary_region)
    total_separation = Integrate(separation_cf, boundary_region)
    total_gap_magnitude = Integrate(gap_magnitude_cf, boundary_region)
    boundary_length = Integrate(CF(1), boundary_region)

    # Average values
    avg_penetration = total_penetration / boundary_length if boundary_length > 0 else 0
    avg_separation = total_separation / boundary_length if boundary_length > 0 else 0
    avg_gap_magnitude = total_gap_magnitude / boundary_length if boundary_length > 0 else 0

    # Create GridFunctions for visualization
    gap_fes = H1(mesh_obj, order=2)

    # Gap normal component
    gap_normal_gf = GridFunction(gap_fes)
    gap_normal_gf.Set(gap_n, definedon=boundary_region)

    # Penetration depth
    penetration_gf = GridFunction(gap_fes)
    penetration_gf.Set(penetration_cf, definedon=boundary_region)

    # Find maximum penetration numerically
    contact_dofs = gap_normal_gf.space.GetDofs(boundary_region)
    gap_values = gap_normal_gf.vec.FV().NumPy()
    boundary_gap_values = [gap_values[i] for i, is_contact in enumerate(contact_dofs) if is_contact]

    max_penetration_numerical = (
        abs(min(boundary_gap_values)) if boundary_gap_values and min(boundary_gap_values) < 0 else 0
    )
    min_gap_value = min(boundary_gap_values) if boundary_gap_values else 0
    max_gap_value = max(boundary_gap_values) if boundary_gap_values else 0

    results = {
        "gap_normal_cf": gap_n,
        "penetration_cf": penetration_cf,
        "separation_cf": separation_cf,
        "gap_magnitude_cf": gap_magnitude_cf,
        "gap_normal_gf": gap_normal_gf,
        "penetration_gf": penetration_gf,
        "avg_penetration": avg_penetration,
        "avg_separation": avg_separation,
        "avg_gap_magnitude": avg_gap_magnitude,
        "max_penetration_numerical": max_penetration_numerical,
        "min_gap_value": min_gap_value,
        "max_gap_value": max_gap_value,
        "boundary_length": boundary_length,
        "total_penetration": total_penetration,
        "total_separation": total_separation,
    }

    return results


# Usage and visualization:
gap_analysis = analyze_contact_gap_comprehensive(contact, mesh, "contact_square")

print("=== Contact Gap Analysis ===")
print(f"Maximum penetration: {gap_analysis['max_penetration_numerical']:.6f}")
print(f"Average penetration: {gap_analysis['avg_penetration']:.6f}")
print(f"Average separation: {gap_analysis['avg_separation']:.6f}")
print(f"Min gap value: {gap_analysis['min_gap_value']:.6f}")
print(f"Max gap value: {gap_analysis['max_gap_value']:.6f}")
print(f"Boundary length: {gap_analysis['boundary_length']:.6f}")
print(f"Total penetration integral: {gap_analysis['total_penetration']:.6f}")

# Visualize the gap distribution
Draw(gap_analysis["gap_normal_gf"], mesh, "gap_normal")
Draw(gap_analysis["penetration_gf"], mesh, "penetration_depth")

In [ ]:
# Method 4: Integration into your simulation loop
def monitor_contact_during_simulation():
    """Add this to your simulation loop for real-time contact monitoring"""

    # Inside your time-stepping loop, after contact.Update():
    gap_vec_master = contact.gap
    n_master = contact.normal
    gap_n_master = InnerProduct(gap_vec_master, n_master)

    # Define penetration coefficient function
    penetration_cf = IfPos(-gap_n_master, -gap_n_master, 0)

    # Calculate contact metrics
    master_boundary = mesh.Boundaries("contact_wall")  # or your boundary name

    # Real-time contact analysis
    current_penetration = Integrate(penetration_cf, master_boundary)
    boundary_length = Integrate(CF(1), master_boundary)
    avg_penetration = current_penetration / boundary_length if boundary_length > 0 else 0

    # Maximum penetration using coefficient function
    max_penetration_cf = penetration_cf  # This gives penetration at each point

    return {
        "avg_penetration": avg_penetration,
        "total_penetration": current_penetration,
        "penetration_cf": penetration_cf,
        "gap_cf": gap_n_master,
    }


# Example integration in your time loop:
# contact_metrics = monitor_contact_during_simulation()
# max_penetration_history.append(contact_metrics['avg_penetration'])